# 03 — DeFi cross-protocol liquidation power-law

**Goal.** Show that three independent DeFi protocols (Aave, Compound, Maker) share the same liquidation-size power-law with α ∈ [1.57, 1.68].

**ETA.** ~3 minutes.

**Source.** Synthetic Pareto with protocol-specific α (offline-reproducible). Real data is in `v4/validation/soc-defi/` of the parent repo (Aave V2 ~12k liquidations, Compound V2 ~8k, Maker ~5k).

**Expected.** Each protocol's α in [1.5, 1.75]. Cross-protocol functional-form match.

In [ ]:
# 1. Synthetic samples for three protocols
import numpy as np

rng = np.random.default_rng(0)
n_per = 5000
samples = {
    "Aave V2":     (1.0 - rng.uniform(0, 1, n_per)) ** (-1.0 / (1.68 - 1)),
    "Compound V2": (1.0 - rng.uniform(0, 1, n_per)) ** (-1.0 / (1.62 - 1)),
    "Maker":       (1.0 - rng.uniform(0, 1, n_per)) ** (-1.0 / (1.57 - 1)),
}
for k, v in samples.items():
    print(f"{k}: n={len(v)}, max={v.max():.2e}")

In [ ]:
# 2. Fit each protocol independently
from soc_pipeline import fit_clauset_powerlaw

fits = {}
for proto, vals in samples.items():
    fit = fit_clauset_powerlaw(vals, name=proto)
    fits[proto] = fit
    print(f"{proto:12s} α = {fit.alpha:.3f} ± {fit.sigma:.3f}  n_tail = {fit.n_tail}")

In [ ]:
# 3. Overlay log-log CCDFs
import matplotlib.pyplot as plt
from soc_pipeline import empirical_ccdf

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
for (proto, vals), c in zip(samples.items(), colors):
    grid, ccdf = empirical_ccdf(vals, n_points=120)
    ax.loglog(grid, ccdf, "o", markersize=3, color=c, alpha=0.6,
              label=f"{proto} (α={fits[proto].alpha:.2f})")
ax.set_xlabel("liquidation size (synthetic units)")
ax.set_ylabel("P(S > s)")
ax.set_title("Cross-protocol liquidation power-law")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 4. Verdict: all three α inside [1.5, 1.75]
from soc_pipeline import verdict_from_alpha_band

for proto, fit in fits.items():
    v = verdict_from_alpha_band(fit.alpha, predicted=(1.5, 1.75), literature=(1.3, 1.9))
    print(f"{proto:12s} α = {fit.alpha:.3f}  verdict: {v}")
print("\nCross-protocol α span — narrow [1.57, 1.68] in the structural-isomorphism dataset")